## Simple Flash attention in Triton

This is basically an online softmax kernel with an extra running variable r_att that is able to calculate attention in one pass.
Block size `B0` represents `k` of length `N0`, `q` of length `N0` and represents `v` of length `N0`.
Sequence length is `T`. Process it `B1 < T` elements at a time.  

$$z_{i} = \sum_{j} \text{softmax}(q_1 k_1, \ldots, q_T k_T)_j v_{j} \text{ for } i = 1\ldots N_0$$


In [1]:
import triton
import torch
from torch import Tensor
import triton.language as tl


def att_spec(q, k, v):
    """This method correctly implements scalar attention"""
    qk = q[:, None] * k[None, :]
    return torch.softmax(qk, dim=1) @ v

def flashatt_explicit(q, k, v):
    """Shows the stabilization the kernel implements"""
    x = q[:, None] * k[None, :]
    x_exp = (x - x.max(1, keepdim=True)[0]).exp() # minus max elem to stablize the softmax operation
    return ((x_exp / x_exp.sum(1, keepdim=True)) * v[None, :]).sum(1)

@triton.jit
def flashatt(q_ptr, k_ptr, v_ptr, z_ptr, T, B0: tl.constexpr):
    """
    Performs simple flash attention with one qkv vector only (no batches)
    online softmax details:
        - Subtracts biggest element to stabalize softmax
        - mutliplies exponent with log2_e which is equivalent to e^x = 2^(log_2(e*x)) (used in softmax) and more efficient
        - The same way as the running denom we can formulate a running attention (r_att) and pull out the denom l out of the sum
    """
    pid_0 = tl.program_id(0) # query
    log2_e = 1.44269504

    # q offset and mask
    q_offset = B0 * pid_0 + tl.arange(0, B0)
    q_mask = q_offset < T

    # load q and rescale to be in log2_e space
    q = tl.load(q_ptr + q_offset, q_mask, other=0.0) * log2_e


    m = tl.full([B0], float("-inf"), dtype=tl.float32) # running max
    l = tl.zeros([B0], dtype=tl.float32)               # running denominator
    r_att = tl.zeros([B0], dtype=tl.float32)           # running attention acc

    for i in tl.range(0, T, B0):

        j_offset = i + tl.arange(0, B0)
        j_mask = j_offset < T

        k = tl.load(k_ptr + j_offset, j_mask, other=0.0)
        v = tl.load(v_ptr + j_offset, j_mask, other=0.0)

        # mask somewhereL
        qk = q[:, None] * k[None,:]
        qk = tl.where(j_mask[None, :], qk, float("-inf"))

        m_new =  tl.maximum(m, tl.max(qk, axis=1))

        alpha = tl.exp2(m - m_new)
        d = tl.exp2(qk - m_new[:, None])

        l = l * alpha + tl.sum(d, axis=1)
        r_att = r_att * alpha + tl.sum(d * v[None, :], axis=1)
        m = m_new

    tl.store(z_ptr + q_offset, r_att / l, q_mask)

correctness check

In [3]:
def flashatt_launch(q, k, v, B0=64):
    T = q.shape[0]
    z = torch.zeros_like(q)
    grid = (triton.cdiv(T, B0),)
    flashatt[grid](q, k, v, z, T, B0=B0)
    return z

torch.manual_seed(0)
T = 200
q, k, v = (torch.randn(T, device="cuda") for _ in range(3))

z = flashatt_launch(q, k, v)
ref = att_spec(q, k, v)

print("max abs err:", (z - ref).abs().max().item())
torch.testing.assert_close(z, ref, rtol=1e-4, atol=1e-4)

max abs err: 1.341104507446289e-07
